Part 1: Data Ingestion & Storage

In [ ]:
import requests
import pandas as pd
import polars as pl
import duckdb
import time

try:
    # 1. Progommatic Donwload of the data and 3.File Organization
    def download_file(url):
        filename = "data/raw/" + url.split('/')[-1]
        with requests.get(url, stream=True) as reference:
            reference.raise_for_status()
            with open(filename, 'wb') as f:
                for chunk in reference.iter_content(chunk_size=8192):
                    f.write(chunk)
        return filename

    filename = download_file('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet')

    df_polars = pl.read_parquet(filename)


    # 2. Data Validation
    required_columns = [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "passenger_count",
        "trip_distance",
        "fare_amount",
        "tip_amount",
        "total_amount",
        "payment_type"
    ]

    missing_columns = [col for col in required_columns if col not in df_polars.columns]

    if not missing_columns:
        print("All required columns are present.")
    else:    
        print("Some required columns are missing.")
        exit(1)
    
    for col in df_polars.columns:
        if "datetime" in col:
            if df_polars[col].dtype != pl.Datetime:
                print(f"Column '{col}' is not in datetime format.")
                exit(1)
    
    print("All datetime columns are in the correct format.")
    
    print(f"Row count: {df_polars.height}")
    
except Exception as e:    
    print(f"An error occurred Validating the data: {e}")
    exit(1)


All required columns are present.
All datetime columns are in the correct format.
Row count: 2964624


Part 2: Data Transformation & Analysis

In [ ]:
rows = df_polars.height
filtered_df = df_polars.filter(
    (pl.col("tpep_pickup_datetime").is_not_null()) &
    (pl.col("tpep_dropoff_datetime").is_not_null()) &
    (pl.col("PULocationID").is_not_null()) &
    (pl.col("DOLocationID").is_not_null()) &
    (pl.col("fare_amount").is_not_null())
).select(pl.all())

filtered = rows - filtered_df.height

if filtered > 0:
    print(f"Rows removed due to null values: {filtered}")
else:
    print(f"No rows removed due to null values.")

filtered_df = df_polars.filter( 
    (pl.col("trip_distance") > 0) &
    (pl.col("fare_amount") >= 0 ) &
    (pl.col("fare_amount") >= 500) 
).select(pl.all())

filtered = filtered - filtered_df.height

if filtered > 0:
    print(f"Rows removed due to invalid trips: {filtered}")
else:
    print(f"No rows removed due to invalid trips.")

filtered_df = filtered_df.filter(
    (pl.col("tpep_pickup_datetime") < pl.col("tpep_dropoff_datetime"))
    ).select(pl.all())

filtered = filtered - filtered_df.height
if filtered > 0:
    print(f"Rows removed due to pickup datetime being after dropoff datetime: {filtered}")
else:
    print("No rows removed due to pickup datetime being after dropoff datetime.")
    
    
engineered = filtered_df.with_columns([
# Calculate trip duration in minutes
((pl.col('tpep_dropoff_datetime') - pl.col('tpep_pickup_datetime'))
.dt.total_seconds() / 60).alias('trip_duration_minutes'),

# Calculate trip speed in miles per hour
(pl.col('trip_distance') / (pl.col('tpep_dropoff_datetime') - pl.col('tpep_pickup_datetime')).dt.total_seconds()).alias('trip_speed_mph'),

# Extract hour of day
pl.col('tpep_pickup_datetime').dt.hour().alias('pickup_hour'),

# Extract day of week
pl.col('tpep_pickup_datetime').alias('pickup_day_of_weekday')   
])





No rows removed due to null values.
No rows removed due to invalid trips.
No rows removed due to pickup datetime being after dropoff datetime.
